# 02d — Background Subtraction for All Videos
Takes 4 user-provided background images (one per camera) and produces
background-subtracted versions of every video in a session.
The result is a simplified foreground-only video per file, ideal for
downstream pose estimation.

## How it works
1. You upload one **background image** per camera (photo of the empty pen).
2. The notebook scans all videos in the session folder.
3. For each video, every frame is difference-subtracted against the
   camera’s background, thresholded, and written to a new video file.
4. Output videos are saved to a `background_subtracted/` folder on Drive.

In [ ]:
# ===== CONFIGURATION =====

GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Google Drive root -- change this to match your Drive structure
DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"  # <-- SET THIS

# Paths (change if your folders are at custom locations)
DRIVE_RAW_VIDEOS = f"{DRIVE_ROOT}/raw_videos"            # Input: session video folders
DRIVE_BACKGROUNDS = f"{DRIVE_ROOT}/backgrounds_sub"      # Input: user-provided background images
DRIVE_OUTPUT = f"{DRIVE_ROOT}/background_subtracted"     # Output: subtracted videos

# Session to process
SESSION = "260608.00000009"  # <-- SET THIS to your session folder name

# Background subtraction parameters
BG_THRESHOLD = 30       # pixel intensity difference threshold
OUTPUT_FPS = None        # set to a number to force output framerate (None = same as input)

# Background filenames (one per camera, placed in DRIVE_BACKGROUNDS/SESSION/)
BG_FILENAME_TEMPLATE = "cam{camera}_bg.jpg"  # e.g. cam1_bg.jpg, cam2_bg.jpg

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
!pip install --quiet opencv-python numpy pandas matplotlib

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm

print("Dependencies installed.")

In [ ]:
from src.io.video_inventory import scan_videos

df = scan_videos(DRIVE_RAW_VIDEOS)
df = df[df["session"] == SESSION].copy()
print(f"\nFound {len(df)} videos for session "{SESSION}"")
df[["filename", "camera", "fps", "frame_count", "width", "height"]]

---
## Step 1 — Provide Background Images

**Before running the next cell**, upload one background image per camera to:
```
{DRIVE_ROOT}/backgrounds_sub/{SESSION}/
    cam1_bg.jpg
    cam2_bg.jpg
    cam3_bg.jpg
    cam4_bg.jpg
```
Each image should be a frame of the **empty pen** (no animal) taken from
the same camera position.

The next cell loads them and shows a preview.

In [ ]:
bg_dir = Path(DRIVE_BACKGROUNDS) / SESSION
bg_dir.mkdir(parents=True, exist_ok=True)

cameras_in_session = sorted(df["camera"].unique())
backgrounds: dict[int, np.ndarray] = {}

for cam in cameras_in_session:
    bg_path = bg_dir / BG_FILENAME_TEMPLATE.format(camera=cam)
    if not bg_path.exists():
        print(f"  MISSING: {bg_path} — upload a background image for camera {cam}")
        continue
    img = cv2.imread(str(bg_path))
    if img is None:
        print(f"  FAILED to load: {bg_path}")
        continue
    backgrounds[cam] = img
    print(f"  Loaded camera {cam}: {bg_path.name} ({img.shape[1]}x{img.shape[0]})")

print(f"\nLoaded {len(backgrounds)} / {len(cameras_in_session)} background images")

# Preview
if backgrounds:
    fig, axes = plt.subplots(1, len(backgrounds), figsize=(5 * len(backgrounds), 4))
    if len(backgrounds) == 1:
        axes = [axes]
    for ax, (cam, bg) in zip(axes, sorted(backgrounds.items())):
        ax.imshow(cv2.cvtColor(bg, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Camera {cam}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

---
## Step 2 — Background-Subtract All Videos

For each video, every frame is processed:
1. Grayscale conversion
2. Absolute difference against the camera’s background
3. Threshold to keep only pixels that changed significantly
4. Write the clean foreground mask as the output frame

Output videos are saved to:
```
{DRIVE_OUTPUT}/{SESSION}/
```

In [ ]:
out_dir = Path(DRIVE_OUTPUT) / SESSION
out_dir.mkdir(parents=True, exist_ok=True)

FOURCC = cv2.VideoWriter_fourcc(*"mp4v")

results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing videos"):
    video_path = Path(DRIVE_RAW_VIDEOS) / row["path"]
    cam = row["camera"]
    stem = Path(row["filename"]).stem
    ext = Path(row["filename"]).suffix

    bg = backgrounds.get(cam)
    if bg is None:
        print(f"  Skipping camera {cam} — no background image")
        results.append({"filename": row["filename"], "status": "skipped (no bg)"})
        continue

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  FAILED to open: {row['filename']}")
        results.append({"filename": row["filename"], "status": "failed to open"})
        continue

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_in = cap.get(cv2.CAP_PROP_FPS)
    total_in = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_out = OUTPUT_FPS if OUTPUT_FPS is not None else fps_in

    out_path = out_dir / f"{stem}_bgsub.mp4"
    writer = cv2.VideoWriter(str(out_path), FOURCC, fps_out, (w, h), isColor=False)

    # Pre-convert background to grayscale for fast diff
    bg_gray = cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY) if bg.ndim == 3 else bg
    if bg_gray.shape != (h, w):
        bg_gray = cv2.resize(bg_gray, (w, h))

    frames_written = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        diff = cv2.absdiff(gray, bg_gray)
        _, thresh = cv2.threshold(diff, BG_THRESHOLD, 255, cv2.THRESH_BINARY)

        # Morphological cleanup: remove small noise, fill gaps
        kernel = np.ones((5, 5), np.uint8)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

        writer.write(thresh)
        frames_written += 1

    cap.release()
    writer.release()

    status = f"ok ({frames_written}/{total_in} frames)"
    print(f"  {row['filename']} → {out_path.name}  [{status}]")
    results.append({"filename": row["filename"], "status": status,
                    "output": str(out_path), "frames": frames_written})

summary_df = pd.DataFrame(results)
print(f"\nDone. Processed {len(summary_df)} videos.")
if len(summary_df) > 0:
    print(summary_df[["filename", "status"]].to_string(index=False))

---
## Step 3 — Preview Results

Compare a raw frame with its background-subtracted version.

In [ ]:
# Pick the first successfully-processed video
done = summary_df[summary_df["status"].str.startswith("ok")]
if len(done) == 0:
    print("No processed videos to preview.")
else:
    row = df[df["filename"] == done.iloc[0]["filename"]].iloc[0]
    video_path = Path(DRIVE_RAW_VIDEOS) / row["path"]
    out_path = done.iloc[0]["output"]
    cam = row["camera"]

    cap = cv2.VideoCapture(str(video_path))
    cap_out = cv2.VideoCapture(out_path)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Show background
    bg_bgr = backgrounds.get(cam)
    if bg_bgr is not None:
        axes[0].imshow(cv2.cvtColor(bg_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Background (Camera {cam})")
    axes[0].axis("off")

    # Pick a middle frame
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    target = total // 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, target)
    cap_out.set(cv2.CAP_PROP_POS_FRAMES, target)

    ret, raw_frame = cap.read()
    ret2, sub_frame = cap_out.read()

    if ret:
        axes[1].imshow(cv2.cvtColor(raw_frame, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Raw frame {target}")
        axes[1].axis("off")

    if ret2:
        axes[2].imshow(sub_frame, cmap="gray", vmin=0, vmax=255)
        axes[2].set_title(f"Background-subtracted")
        axes[2].axis("off")

    cap.release()
    cap_out.release()
    plt.tight_layout()
    plt.show()

    print(f"Input:  {row['filename']}")
    print(f"Output: {out_path}")

---
## Next Steps

The background-subtracted videos are ready for pose estimation.
Proceed to **notebook 03** (Pose Training) or **notebook 04** (Pose Inference).

If you need to reprocess with different parameters (e.g.
`BG_THRESHOLD`), just change the settings in the config cell and
re-run **Step 2**.